### Structured Output
 Models can be requested to provide their response in a format matching a given schema. This is useful for ensuring the output can be easily parsed and used in subsequent processing. LangChain supports multiple schema types and methods for enforcing structured output.

### Pydantic

Pydantic models provide the richest feature set with field validation, descriptions, and nested structures.

In [71]:
import os
from langchain.chat_models import init_chat_model
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")
model=init_chat_model("groq:qwen/qwen3-32b")
model

ChatGroq(output_version=None, profile={'max_input_tokens': 131072, 'max_output_tokens': 16384, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x000002A11CF4F370>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000002A11CF4D4E0>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [72]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    title: str = Field(description="The title of the movie")
    year: int = Field(description="The year the movie was released")
    director: str = Field(description="The director of the movie")
    rating: float = Field(description="The rating of the movie out of 10")

In [73]:
model_with_structure=model.with_structured_output(Movie)
model_with_structure

_ChatModelBinding(bound=ChatGroq(output_version=None, profile={'max_input_tokens': 131072, 'max_output_tokens': 16384, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x000002A11CF4F370>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000002A11CF4D4E0>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': {'properties': {'title': {'description': 'The title of the movie', 'type': 'string'}, 'year': {'description': 'The year the movie was released', 'type': 'integer'}, 'director': {'description': 'The director of the movie', 'type': 'string'}, 'rating': {'description': 'The rating of the movi

In [74]:
response = model_with_structure.invoke(
	"Provide the title, year, director, and rating for the movie Inception."
)
response

Movie(title='Inception', year=2010, director='Christopher Nolan', rating=8.8)

### Message output alongside parsed structure

In [75]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    title: str = Field(..., description="The title of the movie")
    director: str = Field(..., description="The director of the movie")
    release_year: int = Field(..., description="The year the movie was released")

model_with_structure=model.with_structured_output(Movie, include_raw=True)
response = model.invoke("Provide details about the movie Inception.")
response

AIMessage(content='<think>\nOkay, so I need to provide details about the movie Inception. Let me start by recalling what I know about it. It\'s a sci-fi film directed by Christopher Nolan, right? The main actor is Leonardo DiCaprio. The title is "Inception," which I think refers to the concept of planting an idea in someone\'s mind. The movie probably involves some kind of dream-sharing technology because I remember hearing it\'s about entering people\'s dreams.\n\nThe plot might be about a thief who steals information by entering people\'s dreams and then gets a chance to erase his criminal past by performing the inverse: planting an idea instead. That\'s the basic premise from the trailers. The director is definitely Christopher Nolan, known for complex narratives like Memento and Interstellar. The cast includes DiCaprio as Dom Cobb, the protagonist. There\'s also Joseph Gordon-Levitt, who I think plays Arthur, Cobb\'s partner. Ellen Page might be in it too, maybe as Ariadne, the arc

### Neseted Structure

In [76]:
from pydantic import BaseModel, Field

class Actor(BaseModel):
    name: str
    year:str

class MovieDetails(BaseModel):
    title: str
    year: int
    cast: list[Actor]
    genres: list[str]
    budget: float | None = Field(None, description="Budget in millions USD")

response = model_with_structure = model.with_structured_output(MovieDetails)
response

_ChatModelBinding(bound=ChatGroq(output_version=None, profile={'max_input_tokens': 131072, 'max_output_tokens': 16384, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x000002A11CF4F370>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000002A11CF4D4E0>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None), kwargs={'tools': [{'type': 'function', 'function': {'name': 'MovieDetails', 'description': '', 'parameters': {'properties': {'title': {'type': 'string'}, 'year': {'type': 'integer'}, 'cast': {'items': {'properties': {'name': {'type': 'string'}, 'year': {'type': 'string'}}, 'required': ['name', 'year'], 'type': 'object'}, 'type': 'array'}, 'genres': {'items': {'type': 'string'}, 'type': 'ar

### TypedDict

TypedDict provides a simpler alternative using Python's built-in typing, ideal when you don't need runtime validation.

In [77]:
from typing_extensions import TypedDict, Annotated

class MovieDict(TypedDict):
    """A movie with details"""
    title: Annotated[str, ..., "The title of the movie"]
    year: Annotated[int, ..., "The year the movie was released"]
    director: Annotated[str, ..., "The director of the movie"]
    rating: Annotated[float, ..., "The rating of the movie"]

model_withtypedict = model.with_structured_output(MovieDict)
response = model_withtypedict.invoke("Please provide the details of the movie avengers")
response

{'director': 'Joss Whedon', 'rating': 8, 'title': 'Avengers', 'year': 2012}

In [78]:
from pydantic import BaseModel, Field

class Actor(TypedDict):
    name: str
    year:str

class MovieDetails(TypedDict):
    title: str
    year: int
    cast: list[Actor]
    genres: list[str]
    budget: float | None = Field(None, description="Budget in millions USD")

response = model_with_structure = model.with_structured_output(MovieDetails)
response

_ChatModelBinding(bound=ChatGroq(output_version=None, profile={'max_input_tokens': 131072, 'max_output_tokens': 16384, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x000002A11CF4F370>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000002A11CF4D4E0>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None), kwargs={'tools': [{'type': 'function', 'function': {'name': 'MovieDetails', 'description': "dict() -> new empty dictionary\ndict(mapping) -> new dictionary initialized from a mapping object's\n    (key, value) pairs\ndict(iterable) -> new dictionary initialized as if via:\n    d = {}\n    for k, v in iterable:\n        d[k] = v\ndict(**kwargs) -> new dictionary initialized with the name=va

In [79]:
model.profile

{'max_input_tokens': 131072,
 'max_output_tokens': 16384,
 'image_inputs': False,
 'audio_inputs': False,
 'video_inputs': False,
 'image_outputs': False,
 'audio_outputs': False,
 'video_outputs': False,
 'reasoning_output': True,
 'tool_calling': True}

### DataClasses

A data class is a class typically containing mainly data, although there aren't really any restrictions. You create it using the @dataclass decorator

In [82]:
import os
os.environ["OPENAI_API_KEY"]=os.getenv("OPENAI_API_KEY")


In [ ]:
from pydantic import BaseModel, Field
from langchain.agents import create_agent

class ContactInfo(BaseModel):
    """Contact information for a person"""
    name: str = Field(description="The full name of the person")
    email: str = Field(description="The email address of the person")
    phone: str = Field(description="The phone number of the person")

agent = create_agent(
    model = "gpt-5",
    response_format = ContactInfo
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Please provide your contact info from: John Doe, john.doe@example.com, 123-456-7890."}]
})
result
# ContactInfo(name='John Doe', email='john.doe@example.com', phone='123-456-7890')

{'messages': [HumanMessage(content='Please provide your contact info from: John Doe, john.doe@example.com, 123-456-7890.', additional_kwargs={}, response_metadata={}, id='e130a95a-789d-498d-8057-70cccef48258'),
  AIMessage(content='{"name":"John Doe","email":"john.doe@example.com","phone":"123-456-7890"}', additional_kwargs={'parsed': None, 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 228, 'prompt_tokens': 209, 'total_tokens': 437, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 192, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DiwlNkth870Wxe9kQvTQDvSbcB3Gb', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019e58b4-4558-7c80-a50a-d714d4f7b859-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_toke

In [85]:
print(result["structured_response"])

name='John Doe' email='john.doe@example.com' phone='123-456-7890'


In [88]:
## Typedict
from typing_extensions import TypedDict
from langchain.agents import create_agent

class ContactInfo(TypedDict):
    """Contact information for a person"""
    name: str
    email: str 
    phone: str 

agent = create_agent(
    model = "gpt-5",
    response_format = ContactInfo
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john.doe@example.com, 123-456-7890."}]
})
print(result["structured_response"])
# ContactInfo(name='John Doe', email='john.doe@example.com', phone='123-456-7890')

{'name': 'John Doe', 'email': 'john.doe@example.com', 'phone': '123-456-7890'}


In [89]:
## Dataclass

from dataclasses import dataclass
from langchain.agents import create_agent

@dataclass
class ContactInfo:
    """Contact information for a person"""
    name: str
    email: str 
    phone: str
agent = create_agent(
    model = "gpt-5", 
    response_format = ContactInfo
)
result["structured_response"]

{'name': 'John Doe', 'email': 'john.doe@example.com', 'phone': '123-456-7890'}